# Autocorrelación
### Econometría II
**Alejandro Mosiño** — Universidad de Guanajuato

In [ ]:
# Ejecutar solo la primera vez
# install.packages(c("tidyverse", "dynlm", "lmtest", "car", "sandwich", "orcutt"))

## Datos y modelo MCO

Usamos el índice de productividad (`Prodb`) para explicar la remuneración real por hora (`Rcompb`) en Estados Unidos, 1960-2005:

$$
\text{Rcompb}_t = \beta_1 + \beta_2 \, \text{Prodb}_t + u_t
$$

In [ ]:
library(tidyverse)

datos <- read.csv("E204--productivity.csv")

m1 <- lm(Rcompb ~ Prodb, data = datos)
summary(m1)

In [ ]:
datos$e <- residuals(m1)
datos <- ts(datos, start = c(1960), frequency = 1)

## Detección gráfica

Graficamos los residuales contra el tiempo: la presencia de rachas o ciclos sugiere autocorrelación.

In [ ]:
plot(datos[, "e"], ylab = "Residuales", xlab = "Tiempo")
abline(h = 0, col = "#ff6f61")

## El correlograma: FAC y FACP

La función de autocorrelación (FAC) mide la correlación entre $e_t$ y sus propios rezagos:

$$
\hat{r}_k = \frac{\sum_{t=1}^{T-k} (e_t - \bar{e})(e_{t+k} - \bar{e})}{\sum_{t=1}^T (e_t - \bar{e})^2}
$$

In [ ]:
acf(datos[, "e"], lag.max = 10, ci = 0.95)

La función de autocorrelación parcial (FACP) es útil para identificar el orden del proceso autorregresivo de los errores.

In [ ]:
pacf(datos[, "e"], lag.max = 10, ci = 0.95)

## Prueba de Ljung-Box

$$
H_0: \text{no hay autocorrelación de orden } k \qquad H_1: \text{sí la hay}
$$

Paso a paso: el estadístico $Q$ se construye a partir de la FAC estimada.

$$
Q = T(T+2) \sum_{j=1}^k \frac{\hat{r}_j^2}{T-j}
$$

In [ ]:
fac <- acf(datos[, "e"], lag.max = 2, plot = FALSE)$acf[-1]
Tn <- nobs(m1)

Q <- Tn * (Tn + 2) * sum(fac^2 / (Tn - seq_along(fac)))
pchisq(Q, df = 2, lower.tail = FALSE)

Con la función `Box.test`:

In [ ]:
Box.test(datos[, "e"], lag = 2, type = "Ljung-Box")

## Prueba de Breusch-Godfrey

$$
H_0: \mathbb{E}(u_t u_\tau) = 0 \ \forall t \neq \tau \qquad H_1: \mathbb{E}(u_t u_\tau) \neq 0 \ \text{para algún } t \neq \tau
$$

Paso a paso: regresamos $e_t$ sobre las variables explicativas y su propio rezago, y calculamos $BG = T R^2$.

In [ ]:
e_l1 <- c(NA, head(datos[, "e"], -1))

aux_bg <- lm(e ~ Prodb + e_l1, data = datos)
BG <- nobs(aux_bg) * summary(aux_bg)$r.squared
pchisq(BG, df = 1, lower.tail = FALSE)

Con la librería `lmtest`:

In [ ]:
library(lmtest)
bgtest(m1, order = 1)

## Prueba de Durbin-Watson

$$
H_0: \rho = 0 \qquad H_1: \rho \neq 0
$$

Paso a paso: el estadístico $DW$ compara la variación de los residuales entre periodos consecutivos con su varianza total.

$$
DW = \frac{\sum_{t=2}^T (e_t - e_{t-1})^2}{\sum_{t=1}^T e_t^2}
$$

In [ ]:
e <- datos[, "e"]
DW <- sum(diff(e)^2) / sum(e^2)
DW

Con la librería `lmtest`:

In [ ]:
dwtest(m1)

## Corrección: MCGF por Cochrane-Orcutt, AR(1)

Suponemos que los errores siguen un proceso $AR(1)$: $u_t = \rho u_{t-1} + \varepsilon_t$. Estimamos $\rho$ y **transformamos** cada variable (incluida la constante) como $\tilde{z}_t = z_t - \rho z_{t-1}$.

In [ ]:
library(dynlm)

aux_co <- dynlm(e ~ L(e, 1) - 1, data = datos)
r <- coef(aux_co)

datos_co <- datos - r * lag(datos, -1)

m2 <- lm(datos.Rcompb ~ datos.Prodb, data = datos_co)
summary(m2)
dwtest(m2)

Una alternativa equivalente es usar la librería `orcutt`, que aplica el procedimiento de forma iterativa:

In [ ]:
library(orcutt)
cochrane.orcutt(m1)

## Corrección: MCGF por Cochrane-Orcutt, AR(2)

Si suponemos que los errores siguen un proceso $AR(2)$: $u_t = \rho_1 u_{t-1} + \rho_2 u_{t-2} + \varepsilon_t$, la transformación es $\tilde{z}_t = z_t - \rho_1 z_{t-1} - \rho_2 z_{t-2}$.

In [ ]:
aux_co2 <- dynlm(e ~ L(e, 1) + L(e, 2) - 1, data = datos)
r1 <- coef(aux_co2)[1]
r2 <- coef(aux_co2)[2]

datos_co2 <- datos - r1 * lag(datos, -1) - r2 * lag(datos, -2)

m3 <- lm(Rcompb ~ Prodb, data = datos_co2)
summary(m3)
dwtest(m3)

## Corrección: errores estándar robustos de Newey-West

In [ ]:
library(sandwich)
coeftest(m1, vcov = NeweyWest(m1))